In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import pandas as pd
import torch
import gc
import re
import ast
from transformers import (
    BartForConditionalGeneration, BartTokenizer,
    PegasusForConditionalGeneration, PegasusTokenizer,
    T5ForConditionalGeneration, T5Tokenizer,
)

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memorie: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

GPU: True
Device: Tesla T4
Memorie: 14.6 GB


In [13]:
INPUT_PATH = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Total filme: {len(df):,}')

def has_valid_review(r):
    if not isinstance(r, str):
        return False
    try:
        reviews = ast.literal_eval(r)
        return isinstance(reviews, list) and len(reviews) >= 1 and any(len(str(x).split()) >= 10 for x in reviews)
    except:
        return False

mask = (
    df['overview'].notna() &
    (df['overview'].str.split().str.len() >= 30) &
    df['review_texts'].apply(has_valid_review)
)

pool = df[mask].copy()
print(f'Filme cu overview lung + review valid: {len(pool):,}')

selected = pool.sample(10, random_state=99).reset_index(drop=True)
print(f'\n10 filme selectate:')
for i, row in selected.iterrows():
    year = str(row['release_date'])[:4]
    nwords = len(str(row['overview']).split())
    print(f'  {i+1:2}. {row["title"]} ({year}) — overview {nwords} cuv')

Total filme: 40,197
Filme cu overview lung + review valid: 8,020

10 filme selectate:
   1. Across the Wide Missouri (1951) — overview 53 cuv
   2. The Craigslist Killer (2011) — overview 34 cuv
   3. Shrek Forever After (2010) — overview 57 cuv
   4. The Witches (1966) — overview 71 cuv
   5. Mara (2018) — overview 75 cuv
   6. The Spy Gone North (2018) — overview 33 cuv
   7. Stonehearst Asylum (2014) — overview 33 cuv
   8. Knights of the Zodiac (2023) — overview 54 cuv
   9. Loveless (2017) — overview 55 cuv
  10. Mean Machine (2001) — overview 58 cuv


In [14]:
def first_n_sentences(text, n=2):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return ' '.join(s for s in sentences[:n] if s)

def remove_title_prefix(summary, title):
    if not isinstance(summary, str) or not isinstance(title, str):
        return summary
    s = summary.strip()
    t = title.strip()
    if s.lower().startswith(t.lower()):
        result = s[len(t):].lstrip('. ').strip()
        return result if result else s
    if len(t) > 10 and s.lower().startswith(t.lower()[:15]):
        result = s[len(t):].lstrip('. ').strip()
        return result if result else s
    return s

selected['input_with_title'] = selected['title'] + '. ' + selected['overview'].fillna('')
selected['input_no_title']   = selected['overview'].fillna('')
selected['first2_sentences'] = selected['overview'].apply(lambda x: first_n_sentences(str(x), n=2))

print('Date pregatite.')

Date pregatite.


In [15]:
def summarize_bart(texts, tokenizer, model, max_len=40, min_len=15):
    results = []
    for text in texts:
        if not isinstance(text, str) or len(text.split()) < 5:
            results.append('[text prea scurt]')
            continue
        inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True).to(device)
        with torch.no_grad():
            ids = model.generate(
                inputs['input_ids'],
                max_length=max_len, min_length=min_len,
                num_beams=4, early_stopping=True,
                length_penalty=2.0, no_repeat_ngram_size=3,
                forced_bos_token_id=0
            )
        results.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return results

def summarize_pegasus(texts, tokenizer, model, max_len=40, min_len=15):
    results = []
    for text in texts:
        if not isinstance(text, str) or len(text.split()) < 5:
            results.append('[text prea scurt]')
            continue
        inputs = tokenizer(text, return_tensors='pt', max_length=512, truncation=True).to(device)
        with torch.no_grad():
            ids = model.generate(
                inputs['input_ids'],
                max_length=max_len, min_length=min_len,
                num_beams=4, early_stopping=True,
                length_penalty=2.0, no_repeat_ngram_size=3
            )
        results.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return results

def summarize_t5(texts, tokenizer, model, max_len=40, min_len=15):
    results = []
    for text in texts:
        if not isinstance(text, str) or len(text.split()) < 5:
            results.append('[text prea scurt]')
            continue
        input_text = 'summarize: ' + text
        inputs = tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True).to(device)
        with torch.no_grad():
            ids = model.generate(
                inputs['input_ids'],
                max_length=max_len, min_length=min_len,
                num_beams=4, early_stopping=True,
                length_penalty=2.0, no_repeat_ngram_size=3
            )
        results.append(tokenizer.decode(ids[0], skip_special_tokens=True))
    return results

print('Functii definite.')

Functii definite.


In [16]:
print('Incarcare BART-large-cnn')
tok_bart = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
mdl_bart = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(device)
mdl_bart.eval()

ov_bart_notitlu = summarize_bart(selected['input_no_title'].tolist(), tok_bart, mdl_bart, max_len=40, min_len=15)

ov_bart_raw = summarize_bart(selected['input_with_title'].tolist(), tok_bart, mdl_bart, max_len=40, min_len=15)
ov_bart_titlu = [remove_title_prefix(s, t) for s, t in zip(ov_bart_raw, selected['title'].tolist())]

print('\nBART fara titlu — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_bart_notitlu[i]}')

print('\nBART cu titlu + post-procesare — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_bart_titlu[i]}')

del mdl_bart
torch.cuda.empty_cache()
gc.collect()
print('\nBART eliberat.')

Incarcare BART-large-cnn


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]


BART fara titlu — primele 3:
  Across the Wide Missouri: In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Blackfoot woman as a way
  The Craigslist Killer: Based on a true story. Philip Markoff, a charismatic and popular med student at Boston University, leads a double life as a brutal and cruel sexual deviant. He abuses prostitutes he
  Shrek Forever After: A bored and domesticated Shrek pacts with dealmaker Rumpelstiltskin to get back to feeling like a real ogre again. When hes duped and sent to a

BART cu titlu + post-procesare — primele 3:
  Across the Wide Missouri: In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Black
  The Craigslist Killer: Based on a true story. Philip Markoff, a charismatic and popular med student at Boston University, leads a double life as a brutal and cruel sexu

In [17]:
print('Incarcare DistilBART-cnn-12-6')
tok_distil = BartTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
mdl_distil = BartForConditionalGeneration.from_pretrained('sshleifer/distilbart-cnn-12-6').to(device)
mdl_distil.eval()

ov_distil = summarize_bart(selected['input_no_title'].tolist(), tok_distil, mdl_distil, max_len=40, min_len=15)

print('\nDistilBART — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_distil[i]}')

del mdl_distil
torch.cuda.empty_cache()
gc.collect()
print('\nDistilBART eliberat.')

Incarcare DistilBART-cnn-12-6


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



DistilBART — primele 3:
  Across the Wide Missouri:  Flint Mitchell marries a Blackfoot woman as a way to gain entrance into her peoples rich lands, but finds she means more to him than a ticket to good beaver habitat . Flint
  The Craigslist Killer:  Philip Markoff is a charismatic and popular med student at Boston University . He leads a double life as a brutal and cruel sexual deviant . He abuses prostitutes he finds via Craigslist .
  Shrek Forever After:  A bored and domesticated Shrek pacts with dealmaker Rumpelstiltskin to get back to feeling like a real ogre again . When he is sent to a twisted version

DistilBART eliberat.


In [18]:
print('Incarcare PEGASUS-XSum')
tok_pegxsum = PegasusTokenizer.from_pretrained('google/pegasus-xsum')
mdl_pegxsum = PegasusForConditionalGeneration.from_pretrained('google/pegasus-xsum').to(device)
mdl_pegxsum.eval()

ov_pegxsum = summarize_pegasus(selected['input_no_title'].tolist(), tok_pegxsum, mdl_pegxsum, max_len=40, min_len=15)

print('\nPEGASUS-XSum — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_pegxsum[i]}')

del mdl_pegxsum
torch.cuda.empty_cache()
gc.collect()
print('\nPEGASUS-XSum eliberat.')

Incarcare PEGASUS-XSum


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



PEGASUS-XSum — primele 3:
  Across the Wide Missouri: The story of a white man's relationship with a Blackfoot woman in the 19th Century is told through the eyes of three generations of the same family.
  The Craigslist Killer: A medical student who kills women he meets through Craigslist is the focus of a true crime thriller from Boston-based writer and novelist Philip Markoff.
  Shrek Forever After: Shrek the Musical is rated PG and opens in UK cinemas on 17 March 2016.

PEGASUS-XSum eliberat.


In [19]:
print('Incarcare PEGASUS-CNN/DailyMail')
tok_pegcnn = PegasusTokenizer.from_pretrained('google/pegasus-cnn_dailymail')
mdl_pegcnn = PegasusForConditionalGeneration.from_pretrained('google/pegasus-cnn_dailymail').to(device)
mdl_pegcnn.eval()

ov_pegcnn = summarize_pegasus(selected['input_no_title'].tolist(), tok_pegcnn, mdl_pegcnn, max_len=40, min_len=15)

print('\nPEGASUS-CNN — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_pegcnn[i]}')

del mdl_pegcnn
torch.cuda.empty_cache()
gc.collect()
print('\nPEGASUS-CNN eliberat.')

Incarcare PEGASUS-CNN/DailyMail


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



PEGASUS-CNN — primele 3:
  Across the Wide Missouri: life good these much – downannere include first find best good these &re – down says firstYou well only person good these – allowre though In quality her those each In good provide
  The Craigslist Killer: life good these much – downannere include first find best good these – little during her cost founders her need wasn & well
  Shrek Forever After: life good these much industry her give need Electronicing no vital her Kate very need June help press first find many good theseS scale each In good provide direct –With her those being me well

PEGASUS-CNN eliberat.


In [20]:
print('Incarcare T5-large')
tok_t5 = T5Tokenizer.from_pretrained('t5-large')
mdl_t5 = T5ForConditionalGeneration.from_pretrained('t5-large').to(device)
mdl_t5.eval()

ov_t5 = summarize_t5(selected['input_no_title'].tolist(), tok_t5, mdl_t5, max_len=40, min_len=15)

print('\nT5-large — primele 3:')
for i in range(3):
    print(f'  {selected.iloc[i]["title"]}: {ov_t5[i]}')

del mdl_t5
torch.cuda.empty_cache()
gc.collect()
print('\nT5-large eliberat.')

Incarcare T5-large


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


T5-large — primele 3:
  Across the Wide Missouri: in the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho . he marries a black
  The Craigslist Killer: Philip markoff, a charismatic and popular med student at boston university, leads a double life as a brutal and cruel sexual deviant . he abuses
  Shrek Forever After: a bored and domesticated shrek pacts with dealmaker Rumpelstiltskin to get back to feeling like a real ogre again . but when

T5-large eliberat.


In [25]:
def count_words(s):
    return len(str(s).split()) if isinstance(s, str) else 0

def ends_complete(s):
    return str(s).strip()[-1] in '.!?)' if isinstance(s, str) and len(s) > 0 else False

stats_rows = []
for i, row in selected.iterrows():
    for model_name, summary in [
        ('BART + titlu',  ov_bart_titlu[i]),
        ('BART fara titlu',      ov_bart_notitlu[i]),
        ('DistilBART',           ov_distil[i]),
        ('PEGASUS-XSum',         ov_pegxsum[i]),
        ('PEGASUS-CNN/DM',       ov_pegcnn[i]),
        ('T5-large',             ov_t5[i]),
        ('2 propozitii',         row['first2_sentences']),
    ]:
        stats_rows.append({
            'model':   model_name,
            'cuvinte': count_words(summary),
            'complet': ends_complete(summary),
        })

stats_df = pd.DataFrame(stats_rows)
agg = stats_df.groupby('model').agg(
    medie_cuv=('cuvinte', 'mean'),
    complete=('complet', 'sum')
).round(1)
agg['% complete'] = (agg['complete'] / 10 * 100).astype(int)

order = ['BART + titlu', 'BART fara titlu', 'DistilBART', 'PEGASUS-XSum', 'PEGASUS-CNN/DM', 'T5-large', '2 propozitii']
print('STATISTICI OVERVIEW (10 filme, max_length=40 pentru toate):')
print(agg.reindex(order)[['medie_cuv', '% complete']].to_string())
print('\n% complete = rezumate care se termina cu semn de punctuatie final')

STATISTICI OVERVIEW (10 filme, max_length=40 pentru toate):
                 medie_cuv  % complete
model                                 
BART + titlu          28.1          40
BART fara titlu       30.6          10
DistilBART            31.8          40
PEGASUS-XSum          26.6          80
PEGASUS-CNN/DM        32.4           0
T5-large              26.6          20
2 propozitii          45.3         100

% complete = rezumate care se termina cu semn de punctuatie final


In [26]:
print('COMPARATIE DETALIATA\n')
for i, row in selected.iterrows():
    print(f'Film {i+1}: {row["title"]} ({str(row["release_date"])[:4]})')
    print(f'Original ({len(str(row["overview"]).split())} cuv): {str(row["overview"])[:200]}')
    print(f'[1] BART + titlu:    {ov_bart_titlu[i]}')
    print(f'[2] BART fara titlu: {ov_bart_notitlu[i]}')
    print(f'[3] DistilBART:      {ov_distil[i]}')
    print(f'[4] PEGASUS-XSum:    {ov_pegxsum[i]}')
    print(f'[5] PEGASUS-CNN/DM:  {ov_pegcnn[i]}')
    print(f'[6] T5-large:        {ov_t5[i]}')
    print(f'[7] 2 propozitii:    {row["first2_sentences"]}')
    print()

COMPARATIE DETALIATA

Film 1: Across the Wide Missouri (1951)
Original (53 cuv): In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Blackfoot woman as a way to gain entrance into her pe
[1] BART + titlu:    In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Black
[2] BART fara titlu: In the 1830s beaver trapper Flint Mitchell and other white men hunt and trap in the then unnamed territories of Montana and Idaho. Flint marries a Blackfoot woman as a way
[3] DistilBART:       Flint Mitchell marries a Blackfoot woman as a way to gain entrance into her peoples rich lands, but finds she means more to him than a ticket to good beaver habitat . Flint
[4] PEGASUS-XSum:    The story of a white man's relationship with a Blackfoot woman in the 19th Century is told through the eyes of three generations of the